# Evolución Arquitectónica: De GPT-2 al Estándar Moderno (LLaMA / Mistral)

El diseño del Transformer no ha dejado de evolucionar. Mientras que el esqueleto central de autoatención autorregresiva y conexiones residuales se mantiene, la implementación técnica de cada componente individual fue optimizada para maximizar la **eficiencia de memoria (VRAM)**, la **estabilidad numérica** y el **escalado de contexto largo**.

---

### Resumen Comparativo de Componentes

| Componente | GPT-2 Clásico | Arquitectura Moderna (LLaMA / Mistral) | ¿Por qué cambió? |
| :--- | :--- | :--- | :--- |
| **Normalización** | `LayerNorm` (Centrado de media + Varianza) | **`RMSNorm`** | `RMSNorm` elimina el cálculo de la media, reduciendo entre un 10% y 50% el coste computacional sin perder estabilidad. |
| **Embeddings de Posición** | Absolutos fijos / aprendidos (`wpe`) sumados a la entrada | **RoPE (Rotary Position Embedding)** | Permite codificar distancias relativas rotando $Q$ y $K$, facilitando extrapolar contextos a decenas o cientos de miles de tokens. |
| **Atención** | Multi-Head Attention (`MHA`) simétrico | **Grouped-Query Attention (`GQA`)** | Reduce drásticamente el tamaño del **KV-Cache** en inferencia al compartir cabezas de $K$ y $V$ entre varios grupos de $Q$. |
| **Capa MLP / FFN** | Lineal $\to$ `GELU` $\to$ Lineal ($4\times$ expansión) | **`SwiGLU`** ($W_{\text{gate}}, W_{\text{up}}, W_{\text{down}}$) | Introduce compuertas multiplicativas con activación SiLU/Swish, mejorando la convergencia y la capacidad representacional. |
| **Sesgos en Capas** | `bias=True` en proyecciones lineales | **`bias=False`** | Ahorra parámetros innecesarios y mejora la estabilidad numérica en precisión mixta (FP16 / BF16). |

---

## ¿Cómo funciona el Modelo Moderno y por qué cada cambio?

### 1. Pre-RMSNorm: Más velocidad sin calcular medias innecesarias
En lugar de normalizar restando la media y dividiendo entre la desviación estándar, `RMSNorm` simplemente divide el vector entre la raíz del promedio cuadrático ($\text{RMS}$). Empíricamente se demostró que el escalado de varianza es lo único necesario para estabilizar el gradiente, permitiendo ahorrar accesos a memoria y operaciones en GPU.

### 2. RoPE: Rotaciones en lugar de sumas fijas
En lugar de reservar una matriz fija `wpe` de tamaño `[seq_len, embed_dim]`, RoPE opera dinámicamente dentro de la atención. Multiplica los vectores de consulta ($Q$) y clave ($K$) por matrices de rotación basadas en ángulos complejos según la posición del token. Esto permite que el modelo entienda naturalmente qué tan lejos está un token de otro, sin importar la longitud total del documento.

### 3. GQA: Salvando la memoria de inferencia (KV-Cache)
En MHA tradicional, si tenemos 32 cabezas, almacenamos 32 cabezas de $K$ y 32 de $V$ por cada token en la memoria VRAM durante la generación de texto. En GQA, mantenemos 32 cabezas de $Q$ para entender consultas complejas, pero agrupamos las claves y valores en solo 4 u 8 cabezas. Esto reduce la memoria requerida hasta en un $75\%$ sin degradar la calidad de las respuestas.

### 4. SwiGLU: Puertas multiplicativas en el MLP
En lugar de una sola proyección de expansión $W_1$, SwiGLU utiliza dos matrices en paralelo: una proyecta el dato ($W_{\text{up}}$) y la otra calcula un filtro no lineal de compuerta ($W_{\text{gate}}$ con activación SiLU). La multiplicación elemento por elemento actúa como un selector fino de qué información debe fluir, superando el rendimiento de arquitecturas basadas en GELU tradicional.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. HIPERPARÁMETROS (Estilo Arquitectura Moderna)
# ==========================================
n_vocab = 50257  # Tamaño del vocabulario[cite: 1]
embed_dim = 768  # Dimensión de embedding (d_model)[cite: 1]
seq_len = 1024  # Contexto máximo[cite: 1]
n_heads = 12  # Cabezas de Query (Q)[cite: 1]
n_kv_heads = 4  # Cabezas de Key/Value (KV) -> GQA (Ratio 3:1)
n_blocks = 12  # Bloques Transformer apilados[cite: 1]
batch_size = 8  #[cite: 1]


# ==========================================
# 2. COMPONENTE: RMSNorm
# ==========================================
class RMSNorm(nn.Module):

  def __init__(self, dim: int, eps: float = 1e-6):
    super().__init__()
    self.eps = eps
    self.weight = nn.Parameter(torch.ones(dim))

  def forward(self, x):
    # x: [B, T, C]
    # No calcula la media, solo escala por la raíz del promedio cuadrático
    rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    return x * rms * self.weight


# ==========================================
# 3. COMPONENTE: Rotary Position Embedding (RoPE)
# ==========================================
class RoPE(nn.Module):

  def __init__(self, head_dim: int, max_seq_len: int = 2048, theta: float = 10000.0):
    super().__init__()
    # Generar frecuencias para las dimensiones pares
    inv_freq = 1.0 / (
        theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
    )
    self.register_buffer("inv_freq", inv_freq, persistent=False)

    # Precomputar cos y sin para la secuencia completa
    t = torch.arange(max_seq_len, dtype=torch.float)
    freqs = torch.outer(t, inv_freq)
    # [max_seq_len, head_dim // 2] -> duplicar a [max_seq_len, head_dim]
    emb = torch.cat((freqs, freqs), dim=-1)
    self.register_buffer("cos_cached", emb.cos(), persistent=False)
    self.register_buffer("sin_cached", emb.sin(), persistent=False)

  def _rotate_half(self, x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

  def forward(self, q, k):
    # q: [B, n_heads, T, head_dim], k: [B, n_kv_heads, T, head_dim]
    T = q.shape[2]
    cos = self.cos_cached[:T, :].unsqueeze(0).unsqueeze(1)  # [1, 1, T, head_dim]
    sin = self.sin_cached[:T, :].unsqueeze(0).unsqueeze(1)

    q_rot = (q * cos) + (self._rotate_half(q) * sin)
    k_rot = (k * cos) + (self._rotate_half(k) * sin)
    return q_rot, k_rot


# ==========================================
# 4. COMPONENTE: Grouped-Query Attention (GQA)
# ==========================================
class GroupedQueryAttention(nn.Module):

  def __init__(self):
    super().__init__()
    self.n_heads = n_heads
    self.n_kv_heads = n_kv_heads
    self.head_dim = embed_dim // n_heads
    self.num_queries_per_kv = self.n_heads // self.n_kv_heads

    # Proyecciones lineales sin sesgo (bias=False)
    self.q_proj = nn.Linear(
        embed_dim, self.n_heads * self.head_dim, bias=False
    )
    self.k_proj = nn.Linear(
        embed_dim, self.n_kv_heads * self.head_dim, bias=False
    )
    self.v_proj = nn.Linear(
        embed_dim, self.n_kv_heads * self.head_dim, bias=False
    )
    self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    self.rope = RoPE(self.head_dim, max_seq_len=seq_len)

  def forward(self, x):
    B, T, C = x.shape

    # Proyecciones
    q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = (
        self.k_proj(x)
        .view(B, T, self.n_kv_heads, self.head_dim)
        .transpose(1, 2)
    )
    v = (
        self.v_proj(x)
        .view(B, T, self.n_kv_heads, self.head_dim)
        .transpose(1, 2)
    )

    # Aplicar Rotary Position Embeddings a Q y K
    q, k = self.rope(q, k)

    # Repetir K y V si n_kv_heads < n_heads (GQA)
    if self.num_queries_per_kv > 1:
      k = (
          k.unsqueeze(2)
          .repeat(1, 1, self.num_queries_per_kv, 1, 1)
          .view(B, self.n_heads, T, self.head_dim)
      )
      v = (
          v.unsqueeze(2)
          .repeat(1, 1, self.num_queries_per_kv, 1, 1)
          .view(B, self.n_heads, T, self.head_dim)
      )

    # Atención causal escalada optimizada
    out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    # Reensamblar y proyectar salida
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    return self.out_proj(out)


# ==========================================
# 5. COMPONENTE: SwiGLU Feed-Forward Network
# ==========================================
class SwiGLUMLP(nn.Module):

  def __init__(self):
    super().__init__()
    # En SwiGLU el tamaño oculto típico es ~ 8/3 * embed_dim redondeado
    hidden_dim = int(2 * (4 * embed_dim) / 3)

    self.gate_proj = nn.Linear(embed_dim, hidden_dim, bias=False)
    self.up_proj = nn.Linear(embed_dim, hidden_dim, bias=False)
    self.down_proj = nn.Linear(hidden_dim, embed_dim, bias=False)

  def forward(self, x):
    # SwiGLU: (Swish(xW_gate) * xW_up) W_down
    return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


# ==========================================
# 6. BLOQUE TRANSFORMER MODERNO
# ==========================================
class ModernTransformerBlock(nn.Module):

  def __init__(self):
    super().__init__()
    self.norm_1 = RMSNorm(embed_dim)
    self.attn = GroupedQueryAttention()
    self.norm_2 = RMSNorm(embed_dim)
    self.mlp = SwiGLUMLP()

  def forward(self, x):
    # Pre-RMSNorm con flujo residual limpio
    x = x + self.attn(self.norm_1(x))
    x = x + self.mlp(self.norm_2(x))
    return x


# ==========================================
# 7. MODELO COMPLETO (Modern LLM)
# ==========================================
class ModernLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()
    # Solo embedding de tokens (RoPE maneja las posiciones)
    self.wte = nn.Embedding(n_vocab, embed_dim)  #[cite: 1]

    # Bloques Transformer apilados
    self.blocks = nn.Sequential(
        *[ModernTransformerBlock() for _ in range(n_blocks)]
    )

    # Normalización final RMSNorm
    self.norm_final = RMSNorm(embed_dim)

    # Unembedding Head con Weight Tying
    self.lm_head = nn.Linear(embed_dim, n_vocab, bias=False)  #[cite: 1]
    self.lm_head.weight = self.wte.weight  #[cite: 1]

  def forward(self, idx):
    # idx: [B, T]
    x = self.wte(idx)  # [B, T, embed_dim]
    x = self.blocks(x)
    x = self.norm_final(x)
    logits = self.lm_head(x)  # [B, T, n_vocab][cite: 1]
    return logits

  def generate(self, idx, max_new_tokens=50, temperature=1.0):
    for _ in range(max_new_tokens):  #[cite: 1]
      logits = self(idx[:, -seq_len:])  #[cite: 1]
      logits = logits[:, -1, :] / temperature  #[cite: 1]
      probs = F.softmax(logits, dim=-1)  #[cite: 1]
      idx_next = torch.multinomial(probs, num_samples=1)  #[cite: 1]
      idx = torch.cat((idx, idx_next), dim=1)  #[cite: 1]
    return idx  #[cite: 1]


# ==========================================
# 8. PRUEBA Y RESUMEN
# ==========================================
model = ModernLanguageModel().to(device)
data = torch.randint(0, n_vocab, size=(batch_size, seq_len)).to(
    device
)  #[cite: 1]
out = model(data)

print(f"Tamaño de entrada:  {data.shape}")  #[cite: 1]
print(f"Tamaño de salida:   {out.shape}")  #[cite: 1]

Tamaño de entrada:  torch.Size([8, 1024])
Tamaño de salida:   torch.Size([8, 1024, 50257])
